## 필독!!!

<h3> 여기 있는 코드는 절대 실행하지 마십시오. </h3>

눈으로만 보고 이해하시거나

복붙하셔서 실제 linux나 파이썬 환경에서 실행해 주시기 바랍니다.

이 파일은 jupyter 파일입니다.

여기 있는 코드는 모두 jupyter가 아닌 실제 파이썬 및 ROS2 환경에서 사용할 수 있는 코드로 작성하였습니다.

ROS2 코드를 jupyter에서 실행하는 방법이 없는 것은 아니나 별도의 방법이 따로 존재하기 때문에(`jupyter_ws` 참고)

여기 있는 코드를 실행하게 될 경우 일부 오류나 무한루프 등에 빠질 수 있는 위험이 있습니다.

### Extension 패키지 실습

(ros2env 패키지 참고)

ROS2에는 topic, service, action, pkg, run, launch 등 다양한 command가 존재하며,

ros2 -h를 통해 어떤 command가 있는지 확인할 수 있었다.

이번엔 직접 ROS2 기본판에 없는 새로운 command `ros2 env`를 추가해보고,

ROS2 command가 어떤식으로 작동하는지 구체적으로 확인해보는 실습을 해보도록 하겠다.

먼저 src 아래에 ros2env 패키지를 만들어준다.

```bash
cd ~/ros2_ws/src
ros2 pkg create --build-type ament_python ros2env
```

그리고 구조와 파일 내용들은 src/ros2env 안 내용을 참고하여 작성한다.

#### setup.py 및 package.xml 변경사항

##### 1. package.xml

```xml
<depend>ros2cli</depend>
```

이 패키지는 일반 노드가 아닌 ros2 명령어를 만드는 패키지이기 떄문에

ros2cli에 의존한다.

##### 2. setup.py

In [ ]:
entry_points={
        "ros2cli.command": [
            "env = ros2env.command.env:EnvCommand",
        ],
        "ros2cli.extension_point": [
            "ros2env.verb = ros2env.verb:VerbExtension",
        ],
        "ros2env.verb": [
            "list = ros2env.verb.list:ListVerb",
            "set = ros2env.verb.set:SetVerb",
        ],
    }

보통은 `'console_scripts'`로 내용을 넣어줬지만

여기서는 위와 같이 내용을 넣어준다.

하나씩 설명을 하면

```python
"ros2cli.command": [
    "env = ros2env.command.env:EnvCommand",
]
```
ROS2 CLI에 새 명령어를 추가하는 과정.

CLI에 ros2 env를 실행하면

`ros2env.command.env:EnvCommand`를 실행하게 해주는 부분이다.

(ros2env/command/env.py 안의 EnvCommand 클래스를 실행한다고 보면된다.)

```python
"ros2cli.extension_point": [
    "ros2env.verb = ros2env.verb:VerbExtension",
]
```
ros2 env 아래에 붙을 하위 명령어 구조를 만들기 위한 설정이다.

ros2 env 뒤에 붙을 list, set 등의 verb를 만들기 위한 것이라고 보면 된다.

```python
"ros2env.verb": [
    "list = ros2env.verb.list:ListVerb",
    "set = ros2env.verb.set:SetVerb",
],
```

여기서 실제적으로 list와 set과 같은 verb 명령어가 정의된다.

즉 예를 들어 ros2 env list를 실행하면

ros2env/verb/list.py 의 ListVerb 클래스가 실행된다는 것이다.

<br>

여기까지 수정내용이 잘 반영된것이 확인되면(만약 안되어 있으면 넣어주고 저장한다)

이제 빌드를 한 후 `source install/setup.bash`를 해주고

```bash
ros2 -h
```

출력 결과에 env가 등장하는지 확인한 다음

아래 명령어를 하나씩 실행하고 결과가 잘 나오는지 확인한다.

```bash
ros2 env list
ros2 env list -a
ros2 env list -r
ros2 env list -d
ros2 env set ROS_DOMAIN_ID 99
ros2 env set ROS_DOMAIN_ID 0        # 다시 원래 기본값 세팅
```

#### command/env.py

In [ ]:
from ros2cli.command import add_subparsers_on_demand
from ros2cli.command import CommandExtension

class EnvCommand(CommandExtension):
    def add_arguments(self, parser, cli_name):
        self._subparser = parser
        add_subparsers_on_demand(
            parser, cli_name, "_verb", "ros2env.verb", required=False
        )

    def main(self, *, parser, args):
        if not hasattr(args, "_verb"):
            self._subparser.print_help()
            return 0
        extension = getattr(args, "_verb")
        return extension.main(args=args)

기본적인 흐름은 아래와 같다.

`ros2 env set ROS_DOMAIN_ID 30 --verbose --force`와 같은 명령어가 CLI에 입력되었다고 하자.

(이하 모든 코드와 설명은 이 명령어를 기준으로 작성되었다.)

ros2cli는 내부적으로 최상위 parser 객체를 생성한다.

(여기서 parser 객체는 명령어 지도 또는 규칙집 정도로 생각하면 된다.)

(즉 어떤 command와 option이 존재하는지에 대한 정보를 저장하는 객체이다.)

그 후 setup.py의 entry_point 정보를 검색하여 `ros2env.command.env:EnvCommand`를 찾아내고,

EnvCommand 클래스를 로딩한 뒤,

ros2cli는 env 명령을 담당하는 parser 객체를 생성하여 최상위 parser에 등록한다.

그 다음 EnvCommand 안의 `add_arguments()` 함수를 호출한다.

여기서 안에 있던 `add_subparsers_on_demand()`함수를 통해

set, list 등의 verb를 등록할 수 있는 규칙을 추가하고,

사용자가 CLI에 입력한 문자열 `env set ROS_DOMAIN_ID 30 --verbose --force`를 parser가 해석하여 args 객체를 생성한다.

마지막으로 `EnvCommand.main()`과 `SetVerb.main()`이 순서대로 실행된다.

In [ ]:
from ros2cli.command import add_subparsers_on_demand
from ros2cli.command import CommandExtension

기본 내장 패키지 `ros2cli.command`로부터 <command> 명령어를 만들 때 상속받는 클래스 `CommandExtension`를 사용하기 위한 코드이다.

또한 하위 verb를 등록하는 함수 `add_subparsers_on_demand`를 사용하기 위한 코드이다.

In [ ]:
class EnvCommand(CommandExtension):
    def add_arguments(self, parser, cli_name):
        self._subparser = parser
        add_subparsers_on_demand(
            parser, cli_name, "_verb", "ros2env.verb", required=False
        )

`EnVCommand()` : `env`명령어가 실행될 때 생성될 클래스. `CommandExtension`로부터 상속받아야 ros2 command로 사용가능하게 만들어준다.

`add_arguments(self, parser, cli_name)` : `env` 명령어에 대한 규칙을 찾아서 등록해주는 함수.

매개변수 `parser`에는 cli가 env parser를 추가한 parser객체가 들어오고,

`cli_name`에는 실행 command, 즉 여기서는 `ros2 env`가 들어온다.

이 함수명도 ROS2 CLI가 찾아서 호출하는 함수이기 때문에 함수명을 바꾸면 찾을 수 없게 되므로 바꾸면 안된다.

`add_subparsers_on_demand(parser, cli_name, "_verb", "ros2env.verb", required=False)` : 실제적으로 parser에 `env` 명령어에 대한 규칙을 찾아서 parser 객체에 등록하는 함수.

`ros2 env` 뒤에 올 수 있는 Verb들은 `ros2env.verb` Entry Point 그룹에서 검색하며, 사용자가 선택한 Verb 객체는 `args._verb`에 저장된다.

예를 들어 `ros2 env set ...`을 입력하면 `SetVerb` 클래스(ros2env/ros2env/ros2env/verb/set.py)가 생성되어 `args._verb`에 저장되고, 

이후 `EnvCommand.main()`에서 해당 Verb의 `main()` 함수를 호출하여 실제 작업을 수행한다.

`required=False`는 `_verb`의 필수입력여부이다. 즉 cli에 명령어를 입력할 때 set, list 같은 verb를 필수로 입력해야 하는지에 대한 여부를 선택하는 변수이다.

지금은 False이기 때문에 `ros2 env`까지만 입력해도 문제없이 작동하나, 

만약 이걸 True로 설정한 상태에서 `ros2 env`만 cli에 입력했다면 에러가 발생하고 경고문을 터미널에 띄운다.

`EnvCommand()` : `ros2 env` 명령어가 실행될 때 생성되는 클래스이다. `CommandExtension`을 상속받아야 ROS2 CLI에서 Command로 인식되어 `ros2` 명령어로 사용할 수 있다.

`add_arguments(self, parser, cli_name)` : `ros2 env` 명령어에 대한 파싱 규칙을 등록하는 함수이다.

ROS2 CLI가 `EnvCommand`를 로딩한 후 자동으로 호출하며, 함수명을 변경하면 ROS2 CLI가 해당 함수를 찾지 못하므로 정상적으로 동작하지 않는다.

`add_subparsers_on_demand(parser, cli_name, "_verb", "ros2env.verb", required=False)` : `ros2 env`의 하위 명령어(Verb)들을 Parser에 등록하는 함수이다.

`ros2 env` 뒤에 올 수 있는 Verb들은 `ros2env.verb` Entry Point 그룹에서 검색하며, 사용자가 선택한 Verb 객체는 `args._verb`에 저장된다.

예를 들어 `ros2 env set ...`을 입력하면 `SetVerb` 객체가 생성되어 `args._verb`에 저장되고, 이후 `EnvCommand.main()`에서 해당 Verb의 `main()` 함수를 호출하여 실제 작업을 수행한다.

In [ ]:
def main(self, *, parser, args):
    if not hasattr(args, "_verb"):
        self._subparser.print_help()
        return 0
    extension = getattr(args, "_verb")
    return extension.main(args=args)

`main(self, *, parser, args)` : `add_arguments`처럼 ros2cli가 실행할 함수. 마찬가지로 이름을 바꾸면 찾지 못해 실행할 수 없다.

여기서 `args`는 사용자가 입력한 명령어를 Parser가 해석하여 저장한 객체이다. 

예를 들어 `ros2 env set ROS_DOMAIN_ID 30 --verbose --force`를 입력하면 `env_name`, `value`, `verbose`, `force`, `_verb` 등의 값이 저장된다.

`if not hasattr(args, "_verb"):` : args 안에 "_verb"가 있는지 확인한다. 즉 `ros2 env` 뒤의 verb가 있는지 확인하는 코드.

`self._subparser.print_help()` : 안내문 출력 코드. 뒤에 return 0 이 있으므로 verb가 없으면 안내문만 출력하고 그대로 종료.

`extension = getattr(args, "_verb")` : `args`의 `_verb`속성, 즉 `args._verb = SetVerb()`를 가져온다.

`return extension.main(args=args)` : `extension`이 여기선 `SetVerb()`이기 때문에, 이 안의 main 함수에 args를 넣어서 실행하는 코드.